# Neural Network

In [ ]:
!pip install tensorflow
!pip install tensorflow-gpu

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
import numpy as np
import pandas as pd
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import tensorflow.keras as keras

Instructions for updating:
non-resource variables are not supported in the long term


In [ ]:
df = pd.read_csv('mentalHdataset.csv')
df.isnull().sum()
df = df.dropna()
df = df.drop(['Timestamp', 'Age', 'state','Unnamed: 0','comments'] ,axis=1)
df['Gender'] = df['Gender'].replace(['Male', 'Female', 'Other'], [0, 1, 2])
df['Country'] = df['Country'].replace(['United States', 'Israel'], [0, 1])
df = df.replace(('No','Yes','Maybe','Some of them', 'Don\'t know','Not sure'), (0,1,2,2,2,2))
df['work_interfere'] = df['work_interfere'].replace(['Rarely', 'Sometimes', 'Often', 'Never'], [0, 1, 2, 3])
df['Age_cat'] = df['Age_cat'].replace(['(31, 72]', '(18, 31]'], [31, 18])
df.head()

,Gender,Country,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,Age_cat
24,0,0,0,1,1,0,26-100,0,1,1,...,2,0,0,1,1,0,1,2,0,31
25,0,0,0,1,1,1,More than 1000,0,0,1,...,Very easy,1,0,2,1,0,1,0,0,31
33,0,0,0,1,1,1,26-100,1,1,1,...,Very easy,2,0,2,2,2,1,2,0,31
45,1,0,0,1,1,1,26-100,0,1,1,...,Somewhat easy,0,0,2,1,0,0,1,0,31
49,0,0,0,1,1,0,26-100,0,1,1,...,2,2,0,2,1,0,0,2,0,18


In [ ]:
x_vars = df.drop(['treatment', 'self_employed', 'no_employees','remote_work','benefits','seek_help','leave','mental_health_consequence', 'supervisor', 'mental_health_interview', 'obs_consequence' ] ,axis=1)
y_vars = df['treatment']
x_vars.dtypes

Gender                     int64
Country                    int64
family_history             int64
work_interfere             int64
tech_company               int64
care_options               int64
wellness_program           int64
anonymity                  int64
phys_health_consequence    int64
coworkers                  int64
phys_health_interview      int64
mental_vs_physical         int64
Age_cat                    int64
dtype: object

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x_vars,y_vars, train_size = 0.7,random_state=42)

In [ ]:
!pip install sklearn

  Preparing metadata (setup.py) ... done
  Created wheel for sklearn: filename=sklearn-0.0.post11-py3-none-any.whl size=2959 sha256=05dde2719487a497c8a82604bf6a7a7502c3a29a045a3956aa39cbe512cf6afc
  Stored in directory: /root/.cache/pip/wheels/aa/9c/60/f67813603a52fc35057868f1aba0003cc75b72583dcaa2c341
Successfully built sklearn


In [ ]:
from sklearn import preprocessing
stS = preprocessing.StandardScaler()
X_train_centered = stS.fit_transform(x_train)
X_test_centered = stS.fit_transform(x_test)
del x_train, x_test

In [ ]:
print(X_train_centered.shape, y_train.shape)
print(X_test_centered.shape, y_test.shape)

(58, 13) (58,)
(26, 13) (26,)


In [ ]:
np.random.seed(123)
tf.set_random_seed(123)

In [ ]:
y_train_onehot = keras.utils.to_categorical(y_train)
print('First 3 labels: ', y_train[:3])
print('\nFirst 3 labels (one-hot):\n', y_train_onehot[:3])

First 3 labels:  106    1
754    1
493    1
Name: treatment, dtype: int64

First 3 labels (one-hot):
 [[0. 1.]
 [0. 1.]
 [0. 1.]]


In [ ]:
model = keras.models.Sequential()
#The first layer
model.add(keras.layers.Dense(units=50,input_dim=X_train_centered.shape[1],kernel_initializer='glorot_uniform',bias_initializer='zeros',activation='tanh'))
#The second layer
model.add(keras.layers.Dense(units=50,input_dim=50,kernel_initializer='glorot_uniform',bias_initializer='zeros',activation='tanh'))
#The third layer
model.add(keras.layers.Dense(units=y_train_onehot.shape[1],input_dim=50,kernel_initializer='glorot_uniform',bias_initializer='zeros',activation='softmax'))

sgd_optimizer = keras.optimizers.legacy.SGD(lr=0.001, decay=1e-7, momentum=.9)
model.compile(optimizer=sgd_optimizer,loss='categorical_crossentropy')

/usr/local/lib/python3.10/dist-packages/keras/src/optimizers/legacy/gradient_descent.py:114: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


In [ ]:
history = model.fit(X_train_centered, y_train_onehot, batch_size=64, epochs=50,verbose=1,validation_split=0.7)

Train on 17 samples, validate on 41 samples
Epoch 1/50
17/17 [==============================] - 0s 29ms/sample - loss: 0.7801 - val_loss: 0.7369
Epoch 2/50
17/17 [==============================] - 0s 1ms/sample - loss: 0.7782 - val_loss: 0.7353
Epoch 3/50
17/17 [==============================] - 0s 1ms/sample - loss: 0.7746 - val_loss: 0.7329
Epoch 4/50
17/17 [==============================] - 0s 1ms/sample - loss: 0.7695 - val_loss: 0.7299
Epoch 5/50
17/17 [==============================] - 0s 609us/sample - loss: 0.7631 - val_loss: 0.7264
Epoch 6/50
17/17 [==============================] - 0s 641us/sample - loss: 0.7555 - val_loss: 0.7225
Epoch 7/50
17/17 [==============================] - 0s 476us/sample - loss: 0.7470 - val_loss: 0.7182
Epoch 8/50
17/17 [==============================] - 0s 722us/sample - loss: 0.7377 - val_loss: 0.7137
Epoch 9/50
17/17 [==============================] - 0s 620us/sample - loss: 0.7276 - val_loss: 0.7088
Epoch 10/50
17/17 [==========================

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training_v1.py:2335: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates = self.state_updates


17/17 [==============================] - 0s 458us/sample - loss: 0.7060 - val_loss: 0.6987
Epoch 12/50
17/17 [==============================] - 0s 478us/sample - loss: 0.6947 - val_loss: 0.6935
Epoch 13/50
17/17 [==============================] - 0s 743us/sample - loss: 0.6831 - val_loss: 0.6882
Epoch 14/50
17/17 [==============================] - 0s 773us/sample - loss: 0.6713 - val_loss: 0.6829
Epoch 15/50
17/17 [==============================] - 0s 666us/sample - loss: 0.6595 - val_loss: 0.6777
Epoch 16/50
17/17 [==============================] - 0s 763us/sample - loss: 0.6477 - val_loss: 0.6725
Epoch 17/50
17/17 [==============================] - 0s 730us/sample - loss: 0.6358 - val_loss: 0.6673
Epoch 18/50
17/17 [==============================] - 0s 653us/sample - loss: 0.6241 - val_loss: 0.6623
Epoch 19/50
17/17 [==============================] - 0s 1ms/sample - loss: 0.6125 - val_loss: 0.6573
Epoch 20/50
17/17 [==============================] - 0s 576us/sample - loss: 0.6010 - v

In [ ]:
y_train_pred = np.argmax(model.predict(X_train_centered, verbose=0), axis=-1)
print('First 3 predictions: ', y_train_pred[:3])

First 3 predictions:  [1 1 1]


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training_v1.py:2359: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


In [ ]:
correct_preds = np.sum(y_train == y_train_pred, axis=0)
train_acc = correct_preds / y_train.shape[0]
print('Training accuracy: %.2f%%' % (train_acc * 100))

Training accuracy: 81.03%


In [ ]:
y_test_pred = np.argmax(model.predict(X_test_centered, verbose=0), axis=-1)
print('First 3 predictions: ', y_test_pred[:3])

First 3 predictions:  [1 0 1]


In [ ]:
correct_preds = np.sum(y_test == y_test_pred, axis=0)
test_acc = correct_preds / y_test.shape[0]
print('Testing accuracy: %.2f%%' % (test_acc * 100))

Testing accuracy: 61.54%


In [ ]:
from sklearn.metrics import accuracy_score,classification_report
print(classification_report(y_test,y_test_pred))

              precision    recall  f1-score   support

           0       0.22      0.40      0.29         5
           1       0.82      0.67      0.74        21

    accuracy                           0.62        26
   macro avg       0.52      0.53      0.51        26
weighted avg       0.71      0.62      0.65        26



In [ ]:
pd.crosstab(y_test, y_test_pred, rownames = ["Actual"],colnames = ["Predicted"])

Predicted,0,1
Actual,,
0,2,3
1,7,14


In [ ]:
print(round(accuracy_score(y_test, y_test_pred, normalize=True, sample_weight=None),4))

0.6154
